In [4]:
import os
import json
import shutil
import yaml
from ultralytics import YOLO
import torch
import matplotlib.pyplot as plt
import random

In [5]:
# [2] Definizione della classe principale
class YOLOTrainer:
    def __init__(self, base_path):
        self.base_path = os.path.abspath(base_path) 
        self.output_path = os.path.join(self.base_path, 'yolo_dataset')
        
    def train(self, epochs=100, batch_size=16, image_size=800):
        yaml_path = self.base_path + "/data.yaml"
        
        # Inizializza il modello
        print("Inizializzazione modello YOLO...") 
        model = YOLO("yolov8n.pt")        
        
        # Configura il training
        print("Avvio training...")
        results = model.train(
            # Parametri base
            data=yaml_path,
            epochs=epochs,
            imgsz=image_size,
            batch=batch_size,
            device='cuda' if torch.cuda.is_available() else 'cpu',
            
            # Early stopping e salvataggio
            patience=20,
            save=True,
            save_period=10,  # Salva il modello ogni 10 epoche
            
            # Logging
            verbose=True,
            plots=True,      # Genera plots durante il training
            
            # Data augmentation
            augment=True,
            degrees=20.0,    # Rotazione massima
            translate=0.2,   # Traslazione massima
            scale=0.6,       # Range di scaling
            fliplr=0.5,      # Flip orizzontale
            flipud=0.3,      # Flip verticale
            mosaic=1.0,      # Mosaic augmentation
            mixup=0.1,       # Mixup augmentation
            copy_paste=0.1,  # Copy-paste augmentation
            
            # Augmentation del colore
            hsv_h=0.015,     # Hue
            hsv_s=0.7,       # Saturazione
            hsv_v=0.4,       # Valore
            
            # Ottimizzazione
            lr0=0.001,       # Learning rate iniziale
            lrf=0.0001,      # Learning rate finale
            momentum=0.9,    # Momentum per SGD
            weight_decay=0.001,  # Weight decay
            cos_lr=True,     # Cosine learning rate scheduling
            
            # Warmup
            warmup_epochs=5,
            warmup_momentum=0.8,
            warmup_bias_lr=0.1,
            
            # Pesi delle loss
            box=7.5,         # Box loss weight
            cls=0.5,         # Class loss weight
            
            # Altre ottimizzazioni
            label_smoothing=0.1,  # Label smoothing
            overlap_mask=True,    # Mask overlap
            
            # Model EMA
            model_ema=True,      # Exponential Moving Average
            ema_decay=0.9999,
            
            # Multi-scale training
            rect=False,          # Rectangular training
            multi_scale=True,    # Multi-scale training
            
            # Validazione
            val=True,            # Esegui validazione
            close_mosaic=10      # Disabilita mosaic nelle ultime epoche
        )
        
        return model, results

In [ ]:
# [7] Esecuzione del training
# Configura il path del dataset
base_path = 'Dataset/yolo'

# Inizializza e esegui il training
trainer = YOLOTrainer(base_path)
model, results = trainer.train(epochs=100)

# Visualizza i risultati
model.val()